In this notebook, we will document and map every found issue with GDPR and AI act. 

# Notebook 03 — Data Governance & Compliance Audit
## NovaCred Credit Application Governance Audit

> **Executive Summary** — This notebook performs a structured compliance audit of the 500 cleaned credit application records produced by Notebook 01. Every data quality issue identified in the cleaning pipeline is mapped to its corresponding obligations under the **General Data Protection Regulation (GDPR)** and the **EU AI Act**, the two primary regulatory frameworks governing the collection, processing, and automated decision-making of personal data in the European Union. For each issue, this notebook documents the legal basis of the violation, assesses its severity, and provides concrete policy recommendations and remediation actions for NovaCred's Data Governance Officer.

### Regulatory Framework

**GDPR (Regulation 2016/679)** governs how personal data of EU individuals is collected, stored, processed, and protected. In the context of credit applications, virtually every field — name, email, SSN, date of birth, income — constitutes personal data and is subject to GDPR obligations including data minimisation, accuracy, storage limitation, and integrity.

**EU AI Act (Regulation 2024/1689)** classifies credit scoring as a **high-risk AI application** under Annex III. This means NovaCred is legally required to ensure the quality of data used to train and operate its credit scoring model, maintain human oversight over automated decisions, and keep detailed documentation of data quality issues and how they were resolved.

### Scope of This Audit

This audit covers the 15 data quality issues catalogued in Notebook 01, organised across five compliance dimensions:

| Dimension | Issues Covered |
|-----------|---------------|
| Uniqueness | Duplicate records, conflicting SSNs |
| Consistency | Gender coding, date formats |
| Validity | Income type, negative credit history, DTI ratio |
| Completeness | Missing income, DOB, email, SSN, timestamp |
| Accuracy | Age plausibility, timestamp plausibility, cross-field plausibility |
| Timeliness |

> **Note** — This audit uses the cleaned dataset `cleaned_credit_applications.csv` produced by Notebook 01 as its primary input. All flags and audit trail columns created during cleaning are used directly as evidence in this compliance analysis.

In [1]:
import pandas as pd
import regex as re
import datetime
import numpy as np

AUDIT_DATE = datetime.datetime(2026, 3, 1)


df_raw = pd.read_csv('../data/raw_credit_applications.csv')
df_clean = pd.read_csv('../data/cleaned_credit_applications.csv')

## Regulatory Articles Reference

The following articles are cited throughout this audit. They are consolidated here for reference and are not repeated in full in each section.

| Article | Regulation | Title | Summary |
|---------|------------|-------|---------|
| Article 5(1)(b) | GDPR | Purpose Limitation | Personal data collected for one purpose must not be reused for another without the data subject's knowledge. |
| Article 5(1)(c) | GDPR | Data Minimisation | Personal data must be adequate, relevant, and limited to what is necessary for the purposes for which it is processed. |
| Article 5(1)(d) | GDPR | Accuracy | Personal data must be accurate and kept up to date; inaccurate data must be erased or rectified without delay. |
| Article 4(5) | GDPR | Pseudonymisation | Pseudonymisation means processing personal data so it can no longer be attributed to a specific data subject without additional information kept separately. |
| Article 22 | GDPR | Automated Decision-Making | Data subjects have the right not to be subject to decisions based solely on automated processing that produce legal or similarly significant effects. |
| Article 33 | GDPR | Data Breach Notification | In the event of a personal data breach, the controller must notify the supervisory authority within 72 hours. |
| Article 10 | EU AI Act | Data and Data Governance | Training, validation, and testing datasets for high-risk AI systems must be relevant, sufficiently representative, and free of errors to the best extent possible. |
| Article 12 | EU AI Act | Record-Keeping | High-risk AI systems must be capable of automatically recording events while in use, enabling auditability of decisions. |
| Article 14(1) | EU AI Act | Human Oversight | High-risk AI systems must be designed so that natural persons can effectively oversee them during their period of use. |

## Section 1 — Uniqueness

### #1 Duplicate Records & #2 Conflicting SSNs

### Compliance Analysis — Uniqueness

#### Applicable Regulations

**GDPR — Article 5(1)(d) — Accuracy**
Duplicate records and conflicting SSNs both compromise the accuracy of personal data held by NovaCred. Where the same individual appears twice, or where two individuals share a single SSN, at least one record contains data that does not accurately represent reality — violating the obligation to ensure personal data is accurate and kept up to date.

**GDPR — Article 5(1)(c) — Data Minimisation**
Retaining duplicate records means NovaCred holds more data about an individual than is necessary for the purpose of credit assessment, directly contradicting the minimisation principle.

**GDPR — Article 5(1)(b) — Purpose Limitation**
Where a single individual has submitted two separate applications, data collected for the first application must not be silently reused or cross-referenced for the second without the applicant's knowledge.

**GDPR — Article 33 — Data Breach Notification**
If two different individuals share the same SSN and identity fraud is confirmed, NovaCred is legally obligated to notify the relevant supervisory authority within 72 hours.

**EU AI Act — Article 10 — Data Governance**
Corrupted identity data — whether through duplication or SSN conflicts — must not be used in any credit scoring model. High-risk AI systems must be trained on data that is free from errors to the best extent possible.

---

#### Issue Breakdown

**#1 — Duplicate Records** — The same individual appears more than once in the dataset. This can occur due to a data pipeline error or because an applicant submitted multiple times. Duplicate records risk causing the same person to be assessed twice under different conditions, producing inconsistent credit outcomes and overweighting those individuals in model training.

**#2 — Conflicting SSNs** — Three distinct cases were identified. In Case A, an exact row duplicate shares the same `_id` and name — a pure pipeline error with no SSN conflict. In Case B, the same person submitted two applications under different IDs, raising purpose limitation concerns. In Case C — the most serious — two different individuals share the same SSN, indicating either a data entry error or identity fraud. At least one record is incorrect, and the case must be treated as a potential data breach until confirmed otherwise.

---

#### Consequences if Unresolved
- Duplicate records may lead to inconsistent credit decisions for the same individual and distort model training by overrepresenting certain applicants.
- A shared SSN between two different individuals means at least one credit decision was made on incorrect identity data, which cannot be legally defended.
- If identity fraud is confirmed in Case C, failure to notify the supervisory authority within 72 hours constitutes a direct breach of GDPR Article 33.
- Corrupted identity records used in model training would introduce incorrect signals into the credit scoring model, potentially affecting predictions for the wider applicant population.

---

#### Recommended Actions
- **Data Engineering Team**: Implement a unique constraint on `_id` at the point of data ingestion — reject any application whose ID already exists in the system.
- **Data Engineering Team**: Schedule periodic deduplication audits on the live database to catch any duplicates that slip through.
- **Governance Officer**: Both records in Case C must be immediately excluded from model training and any automated credit decision. The case must be escalated to NovaCred's legal team and, if fraud is confirmed, reported to the relevant supervisory authority.
- **Governance Officer**: The two applications in Case B must be manually reviewed to determine whether the second was intentional or a system error, and the applicant notified accordingly.

## Section 2 - Consistency

### #3 Gender Coding & #4 Date Formats

### Compliance Analysis — Consistency

#### Applicable Regulations

**GDPR — Article 5(1)(d) — Accuracy**
Inconsistent encoding of gender values and mixed date formats both compromise the accuracy of personal data. When the same value is stored in multiple formats, the system cannot reliably interpret it — meaning NovaCred may be operating on data that does not correctly represent the individual, in violation of the accuracy principle.

**EU AI Act — Article 10 — Data Quality**
High-risk AI systems must use training data that is consistent and free from encoding errors. Both issues identified in this section directly violate this requirement — a model trained on inconsistently encoded gender or misinterpreted dates of birth would receive contradictory or incorrect inputs, producing unreliable credit decisions.

---

#### Issue Breakdown

**#3 — Inconsistent Gender Coding** — The `gender` field contains four distinct encodings of the same two values: `Male`, `M`, `Female`, and `F`. Any system reading this field — including the credit scoring model — may treat `M` and `Male` as different categories, introducing silent encoding errors into automated decisions. Statistical analysis of gender distribution across approvals and rejections would also produce incorrect results, undermining fairness reporting obligations.

**#4 — Mixed Date Formats** — Date of birth is stored across four different formats (`YYYY-MM-DD`, `DD/MM/YYYY`, `MM/DD/YYYY`, `YYYY/MM/DD`), with 5 records in an unrecognised `UNKNOWN` format. The risk of misinterpretation is concrete — `03/04/1990` could be read as March 4th or April 3rd depending on the assumed format, meaning the system may be processing an incorrect date of birth for a subset of applicants. The 5 `UNKNOWN` records have dates that could not be reliably interpreted at all.

---

#### Consequences if Unresolved
- The credit scoring model may produce different outcomes for applicants encoded as `M` vs `Male`, constituting an unintended encoding bias that cannot be detected without a full audit of model inputs.
- Misinterpreted dates of birth would cause applicants to be assessed as older or younger than they are, potentially affecting credit eligibility in ways that cannot be traced back to an encoding error.
- The 5 `UNKNOWN` format records have effectively unknown birth dates despite appearing to contain data — any age-dependent feature derived from these records is unreliable.
- Fairness reporting on gender distribution across approvals and rejections would produce incorrect results, undermining NovaCred's ability to demonstrate equitable outcomes.

---

#### Recommended Actions
- **Data Engineering Team**: Implement input validation at ingestion — the `gender` field should only accept a defined set of values (e.g. `Male`, `Female`). Any abbreviation or non-standard value should be rejected or automatically standardised before the record enters the database.
- **Data Engineering Team**: Enforce ISO 8601 (`YYYY-MM-DD`) as the only accepted date format at the point of data ingestion. Any other format should be rejected with a clear error message.
- **Data Engineering Team**: The 5 records with `UNKNOWN` date format must be manually reviewed — their date of birth cannot be used until the correct value is confirmed with the applicant.

# Section 3 - Validity

## #5 Income type, #6 Negative credit history, #7 DTI ratio

### Compliance Analysis — Validity

#### Applicable Regulations

**GDPR — Article 5(1)(d) — Accuracy**
All three validity issues — income stored as text, negative credit history, and an impossible DTI ratio — constitute accuracy violations under this article. In each case, the field contains a value that is either technically incorrect (wrong data type) or factually impossible (negative months, ratio above 1.0), meaning NovaCred is processing and potentially making credit decisions based on inaccurate personal data.

**EU AI Act — Article 10(3) — Data and Data Governance**
Credit scoring is classified as a high-risk AI application under Annex III of the EU AI Act. All three issues directly compromise the quality of data used to train and operate the model — a type error in income, an impossible credit history value, and an out-of-range DTI ratio would each corrupt the model's inputs and potentially lead to incorrect or discriminatory credit decisions.

**EU AI Act — Article 14(1) — Human Oversight**
Any automated correction applied to these fields — such as the absolute value fix for negative credit history or the division by 10 for the DTI ratio — must be logged, documented, and formally approved by the Governance Officer before the corrected records are used in any model training or credit decision.

---

#### Issue Breakdown

**#5 — Income Stored as Text** — Storing `"55000"` as a string instead of a number means the value cannot be reliably used in numerical computation. A system that fails to convert it correctly would either crash or silently treat the income as null, leading to an incorrect credit assessment. Applicants with text-encoded income risk being assessed as having no income, resulting in a wrongful rejection.

**#6 — Negative Credit History Months** — A value of `-10 months` is factually impossible — credit history cannot be negative. This is either a data entry error or a system bug. If fed directly into the credit scoring model, it would be interpreted as an extremely poor credit profile, leading to an unjust rejection of the affected applicant.

**#7 — Impossible Debt-to-Income Ratio** — A DTI of `1.85` implies the applicant's debt is 185% of their income, which is mathematically impossible as a ratio. The most likely explanation is a decimal point error (`1.85` instead of `0.185`). If uncorrected, the model would treat this applicant as catastrophically over-indebted, virtually guaranteeing an incorrect rejection.

---

#### Consequences if Unresolved
- Applicants affected by these errors could receive incorrect loan rejections based on corrupted data, constituting unfair automated decisions under **GDPR Article 22**.
- The credit scoring model trained on this data would learn distorted relationships between features and creditworthiness, undermining the reliability of all future decisions.
- NovaCred would be in breach of its EU AI Act obligations regarding data quality for high-risk AI systems.

---

#### Recommended Actions
- **Data Engineering Team**: Enforce strict type validation on `annual_income` at ingestion — reject any value that cannot be cleanly cast to a number.
- **Data Engineering Team**: Implement range validation at ingestion — `credit_history_months` must be a non-negative integer and `debt_to_income` must be between `0.0` and `1.0`. Any value outside these ranges should be rejected and flagged for manual review.
- **Governance Officer**: Formally approve and document the corrections applied in Notebook 01 (`abs()` for credit history, `÷10` for DTI) before the affected records are used in any model training or credit decision.

# Section 4 - Completeness
## #8 Missing income, DOB, email, SSN, timestamp

### Compliance Analysis — Completeness

#### Applicable Regulations

**GDPR — Article 5(1)(d) — Accuracy**
Missing personal data fields — date of birth, email, SSN, and processing timestamp — mean that NovaCred is operating with incomplete records. An incomplete record cannot be considered accurate, as the absence of a required field is itself a data quality defect that affects the reliability of any decision made on that record.

**GDPR — Article 5(1)(c) — Data Minimisation**
While data minimisation typically concerns collecting too much data, it also applies in reverse — if a field is mandatory for the purpose of credit assessment, its absence means the record is inadequate for that purpose and should not be processed further.

**GDPR — Article 22 — Automated Individual Decision-Making**
Seven applicants were approved for loans despite having missing email addresses, and four were approved despite missing SSNs. This means NovaCred made legally significant automated credit decisions on records with incomplete identity data — a direct violation of the principle that automated decisions must be based on accurate and complete information.

**EU AI Act — Article 10(3) — Data and Data Governance**
Missing values in key fields such as income, date of birth, and SSN directly compromise the completeness requirement for high-risk AI training data. A credit scoring model trained on records with missing identity or financial fields will learn from an incomplete picture of the applicant population, potentially introducing systematic bias into its predictions.

---

#### Issue Breakdown

**#8 — Missing Date of Birth** — Date of birth is PII and cannot be imputed or fabricated. The 4 affected records have been flagged with `dob_missing = True` and their `date_of_birth` and `age` fields preserved as `NaT` and `NaN` respectively. These records must not be used in any age-dependent feature engineering until the correct value is confirmed with the applicant.

**#9 — Missing Email Address** — Email is a primary contact channel for identity verification and regulatory notifications. 7 records have missing email addresses, of which 6 were approved for loans. NovaCred cannot fulfil its GDPR notification obligations — including data breach notification under Article 33 or responding to Subject Access Requests under Article 15 — for applicants whose email is unknown.

**#10 — Missing SSN** — SSN is the primary identity verification field in this dataset. 5 records have no SSN, meaning NovaCred cannot uniquely verify the identity of these applicants. Approving a loan without a verified identity is a significant compliance and fraud risk.

**#11 — Missing Processing Timestamp** — 400 records have no processing timestamp. While the loan decision itself remains valid, the absence of a timestamp means NovaCred cannot reconstruct when these decisions were made — undermining the audit trail required by the EU AI Act for high-risk automated decisions.

**#12 — Missing Income** — Unlike PII fields, missing income was imputed with the dataset median ($81,000) and flagged with `income_imputed = True`. While this is an acceptable pragmatic fix, imputed values introduce uncertainty into the model's income feature and must be treated with caution in any downstream analysis.

---

#### Consequences if Unresolved
- Applicants with missing PII could receive credit decisions that cannot be legally defended or audited, exposing NovaCred to liability under GDPR Article 22.
- The absence of processing timestamps for 400 records means NovaCred cannot demonstrate compliance with its own decision-making process — a direct breach of EU AI Act transparency requirements.
- A credit scoring model trained on records with missing identity fields may produce biased predictions for applicants with incomplete profiles.

---

#### Recommended Actions
- **Data Engineering Team**: Make `date_of_birth`, `email`, `ssn`, and `processing_timestamp` mandatory fields at ingestion — reject any application that does not provide all four values.
- **Data Engineering Team**: Implement a mandatory timestamp write-back at the point of every loan decision — no decision should be recorded without an associated timestamp.
- **Governance Officer (this report)**: The 4 records with missing DOB, 7 with missing email, and 5 with missing SSN are hereby flagged for mandatory review before use in any model training cycle. The median imputation applied to the 5 missing income records is formally approved, on the condition that `income_imputed = True` is used as a feature flag in all downstream models to signal imputed values.

---

#### A Note on Pseudonymisation

Several of the missing PII fields identified in this section — email, SSN, and date of birth — are also candidates for **pseudonymisation** under GDPR Article 4(5), which defines pseudonymisation as the processing of personal data in such a way that the personal data can no longer be attributed to a specific data subject without the use of additional information.

In the context of this dataset, pseudonymisation would involve replacing direct identifiers such as SSN and email with irreversible tokens or hashed values before the data is passed to the data science team for model training. This would ensure that the model never has access to raw PII, reducing NovaCred's exposure in the event of a data breach. A full pseudonymisation implementation will be addressed in the next section.

## Privacy protection

*GDPR Article 4(5)*: Pseudonymization means processing personal data so that it can no longer be attributed to a specific data subject without additional information, provided that such additional information is kept separately and is subject to technical and organizational measures

In [2]:
## print(df_clean.columns)
df_clean.iloc[:10, :20].columns

Index(['_id', 'full_name', 'email', 'ssn', 'ip_address', 'gender',
       'date_of_birth', 'age', 'zip_code', 'annual_income', 'income_imputed',
       'credit_history_months', 'credit_history_months_flag', 'debt_to_income',
       'dti_flag', 'savings_balance', 'loan_approved', 'interest_rate',
       'approved_amount', 'rejection_reason'],
      dtype='object')

## Pseudonymization

### Tokenization of full name and email hashing with salt 

In [3]:
import hashlib
import os
import random
import string

# ── Tokenization for full_name ──────────────────────────────
token_lookup = {}  # store this securely, like the salt!

def tokenize(value):
    if value not in token_lookup:
        token = 'TKN_' + ''.join(random.choices(string.digits, k=6))
        token_lookup[value] = token
    return token_lookup[value]

df_clean['full_name'] = df_clean['full_name'].apply(tokenize)

# Save lookup table securely
import json
with open('token_lookup.json', 'w') as f:
    json.dump(token_lookup, f)

# ── Hashing + salt for email ────────────────────────────────
salt = os.urandom(16).hex()

def hash_with_salt(value, salt):
    salted = (salt + str(value)).encode('utf-8')
    return hashlib.sha256(salted).hexdigest()

df_clean['email'] = df_clean['email'].apply(lambda x: hash_with_salt(x, salt))

with open('salt.key', 'w') as f:
    f.write(salt)

In [4]:
df_clean[['_id','full_name','email']].head()

,_id,full_name,email
0,app_200,TKN_905902,c5e26f450af27ab8c4df90a9992d34e962fcd7e8603853...
1,app_037,TKN_451201,78a73ea33f2ac43378b227b701f6dc4a075b145dd4f607...
2,app_215,TKN_536334,189ad918bd469aec8f24a2359d9cf53c21232d784e4cb3...
3,app_024,TKN_316527,5743dfcbf5f5c32908f05f6d4b0f190115a2c735036d81...
4,app_184,TKN_036972,6aadf96302e9d40b2f34c32ddea898234fe184b02166bf...


For **full name**, we applied **tokenization**, replacing each unique name with a randomly 
generated token (e.g., `TKN_482910`), stored in a separate lookup table. This approach was 
chosen because it provides the strongest pseudonymization guarantee, and with it, there is no mathematical 
relationship between the original value and the token, making reverse engineering impossible 
without the lookup table. The key weakness is that the lookup table itself becomes a high-value 
target: if stolen alongside the dataset, re-identification is trivial.

For **email address**, we applied **hashing with salt** (SHA-256). Unlike plain hashing, 
the salt ensures that an attacker cannot pre-compute a dictionary of known email hashes 
to match against our dataset. Email was chosen for this approach rather than tokenization 
because hashing requires no lookup table to maintain, reducing operational risk. The remaining 
weakness is that hashing is deterministic, the same email with the same salt always produces 
the same hash, meaning that if the salt is compromised, all records become vulnerable 
simultaneously.

Both techniques satisfy GDPR pseudonymization requirements under Article 4(5), provided that 
the token lookup table and salt are stored separately from the dataset, access-controlled, and never co-located with the pseudonymised data itself.

### K-Anonymization — SSN and IP Address

In [5]:
# ── SSN: suppress first two groups (***-**-6789) ────────────
def suppress_ssn(value):
    parts = str(value).split('-')
    if len(parts) == 3:
        return f"***-**-{parts[2]}"
    return "***-**-****"  # fallback if format is unexpected

df_clean['ssn'] = df_clean['ssn'].apply(suppress_ssn)

# ── IP Address: mask last two octets (192.168.*.*) ──────────
def suppress_ip(value):
    parts = str(value).split('.')
    if len(parts) == 4:
        return f"{parts[0]}.{parts[1]}.*.*"
    return "*.*.*.*"  # fallback

df_clean['ip_address'] = df_clean['ip_address'].apply(suppress_ip)

# ── Processing timestamp: truncate to date only ─────────────
df_clean['processing_timestamp'] = pd.to_datetime(df_clean['processing_timestamp']).dt.date

In [6]:
df_clean[['_id','ssn','ip_address','processing_timestamp']]

,_id,ssn,ip_address,processing_timestamp
0,app_200,***-**-4340,192.168.*.*,2024-01-15
1,app_037,***-**-4784,10.1.*.*,NaT
2,app_215,***-**-5178,10.240.*.*,NaT
3,app_024,***-**-1833,192.168.*.*,NaT
4,app_184,***-**-2475,172.29.*.*,2024-01-15
...,...,...,...,...
495,app_468,***-**-6099,172.31.*.*,NaT
496,app_192,***-**-8002,172.29.*.*,2024-01-15
497,app_234,***-**-8731,10.143.*.*,NaT
498,app_306,***-**-8131,10.26.*.*,NaT


## K-Anonymization Strategy: SSN and IP Address

For **SSN**, we applied **partial suppression**, keeping only the last four digits 
(e.g., `***-**-6789`). This approach is the standard for SSN anonymization, 
familiar from US regulatory practice, and ensures that the most identifying segment - 
the area and group numbers - is removed while preserving just enough structure for 
legitimate record linkage if required. The weakness is that the last four digits alone, 
while insufficient for full re-identification, can still contribute to linkage attacks 
when combined with other quasi-identifiers such as date of birth.

For **IP address**, we masked the last two octets (e.g., `192.168.*.*`), retaining only 
the network prefix. This reduces granularity to the subnet level, making it impossible 
to identify an individual device or household while preserving coarse geographic and 
network-level information useful for fraud detection and audit purposes. The 
risk is that in small or corporate networks, even a subnet prefix may be sufficient to 
narrow down a user to a small group.


We truncated the timestamp to **date precision only** (e.g., 
`2024-03-15 14:32:00` → `2024-03-15`). This preserves temporal context for legitimate 
analytical purposes, such as trend analysis, seasonal patterns in credit applications, 
or audit trails, while eliminating the granularity required for session-level 
re-identification. 

All three transformations achieve **k-anonymization** in the sense that each retained value 
is shared by multiple individuals, preventing unique identification from these fields 
alone. However, true k-anonymization guarantees require verifying that no combination 
of retained fields uniquely identifies any record.

### Quasi-Identifiers - ZIP, gender, birth date

In [7]:
# ── ZIP code: generalize to 3 digits ────────────────────────
df_clean['zip_code'] = df_clean['zip_code'].astype(str).str[:3] + '**'

# ── Age: drop (redundant with date_of_birth) ────────────────
try:
    df_clean.drop(columns=['age'], inplace=True)
except:
    print('Age column already dropped')

# ── Date of birth: keep year only ───────────────────────────
df_clean['date_of_birth'] = pd.to_datetime(df_clean['date_of_birth']).dt.year

# ── Gender: keep as-is (low entropy, acceptable risk) ───────
# no change needed

In [8]:
df_clean[['_id','zip_code','date_of_birth']]

,_id,zip_code,date_of_birth
0,app_200,100**,2001.0
1,app_037,100**,1992.0
2,app_215,100**,1989.0
3,app_024,100**,1983.0
4,app_184,100**,1999.0
...,...,...,...
495,app_468,100**,1999.0
496,app_192,100**,1985.0
497,app_234,100**,1976.0
498,app_306,902**,1978.0


Quasi-identifiers are fields that, while not directly identifying on their own, can 
uniquely identify an individual when combined. Following Sweeney (2000), who demonstrated 
that 87% of Americans can be uniquely re-identified using only ZIP code, gender, and date 
of birth, these fields required careful treatment.

For **ZIP code**, we applied **generalization to 3 digits** (e.g., `12345` → `123**`), 
reducing geographic precision to the regional level while preserving enough structure 
for aggregate geographic analysis. Full suppression was rejected as it would eliminate 
useful signal for credit risk modeling across regions.

For **date of birth**, we retained **only the birth year** (e.g., `1990-05-12` → `1990`),
removing month and day which are the primary contributors to uniqueness. This preserves 
age-related analytical value while substantially reducing re-identification risk.

The **age** column was dropped entirely, as it is a derived field calculated directly 
from date of birth. Retaining it alongside a generalized date of birth would be 
contradictory — a precise age value would undermine the generalization already applied 
to the source field.

**Gender** was retained without modification. As a protected attribute under EU 
non-discrimination law and a key variable for AI Act fairness obligations, gender must 
remain available for bias auditing and equitable outcome analysis. Suppressing it would 
not only eliminate critical analytical utility but would also make it impossible to 
detect or correct discriminatory patterns in credit decision outcomes.

### Differential privacy - adding noise to annual income, credit_history_months, debt_to_income, savings_balance, interest_rate, approved_amount

In [9]:
def laplace_interval_str(col, sensitivity, epsilon):
    scale = sensitivity / epsilon
    noise = np.abs(np.random.laplace(0, scale, len(col)))
    return (col - noise).round(0).astype(str) + ' - ' + (col + noise).round(0).astype(str)

epsilon = 1.0

df_clean['annual_income']         = laplace_interval_str(df_clean['annual_income'], 10000, epsilon)
df_clean['savings_balance']       = laplace_interval_str(df_clean['savings_balance'], 5000, epsilon)
df_clean['approved_amount']       = laplace_interval_str(df_clean['approved_amount'], 2000, epsilon)
df_clean['debt_to_income']        = laplace_interval_str(df_clean['debt_to_income'], 0.02, epsilon)
df_clean['interest_rate']         = laplace_interval_str(df_clean['interest_rate'], 0.005, epsilon)
df_clean['credit_history_months'] = laplace_interval_str(df_clean['credit_history_months'], 3, epsilon)

In [11]:
df_clean[['_id','annual_income', 'credit_history_months', 'debt_to_income', 'savings_balance', 'interest_rate', 'approved_amount']].head()

,_id,annual_income,credit_history_months,debt_to_income,savings_balance,interest_rate,approved_amount
0,app_200,68677.0 - 77323.0,22.0 - 24.0,0.0 - 0.0,30004.0 - 32420.0,nan - nan,nan - nan
1,app_037,72292.0 - 83708.0,50.0 - 52.0,0.0 - 0.0,8856.0 - 26974.0,nan - nan,nan - nan
2,app_215,53942.0 - 68058.0,39.0 - 43.0,0.0 - 0.0,34160.0 - 41658.0,4.0 - 4.0,56504.0 - 61496.0
3,app_024,91516.0 - 114484.0,69.0 - 71.0,0.0 - 0.0,-3436.0 - 3436.0,4.0 - 4.0,33539.0 - 34461.0
4,app_184,42667.0 - 71333.0,13.0 - 15.0,0.0 - 0.0,30707.0 - 32819.0,nan - nan,nan - nan


#### Numerical Perturbation via Differential Privacy: Laplace Interval Mechanism

For continuous numerical fields, we applied the **Laplace mechanism** from differential 
privacy theory, replacing each exact value with an interval representing the range of 
plausible true values. Rather than storing a single point estimate, each value 
is expressed as `[value - noise, value + noise]`, where the noise magnitude is drawn 
from a Laplace distribution parameterized by sensitivity and epsilon (ε).

The **sensitivity** parameter reflects the maximum realistic contribution of a single 
individual's record to the field — for example, ±€10,000 for annual income, ±3 months 
for credit history, or ±0.02 for debt-to-income ratio. The **epsilon parameter** was set 
to ε = 1.0, representing a moderate privacy budget that balances re-identification 
protection with analytical utility. Lower values of ε would provide stronger privacy 
guarantees at the cost of wider, less useful intervals.

This approach was applied to: `annual_income`, `savings_balance`, `approved_amount`, `debt_to_income`, `interest_rate`, and `credit_history_months`. The `age` column was already dropped during quasi-identifier generalisation and is therefore excluded. Binary and categorical fields, as well as flag columns, were excluded as alteration would corrupt their logical meaning. The interval representation communicates inherent uncertainty to downstream consumers of the dataset, preventing false precision and discouraging exact value reconstruction.

### Coverage of spending behaviour

In [2]:
TRESHOLD_ALCOHOL = 200
TRESHOLD_GAMBLING = 100

df_clean['red_flag_spending'] = (
    (df_clean['gambling_spend'] > THRESHOLD_GAMBLING) |
    (df_clean['had_adult_entertainment_raw'] == True) |
    (df_clean['alcohol_spend'] > THRESHOLD_ALCOHOL) 
).astype(int)

NameError: name 'df_clean' is not defined

#### Applicable Regulations

**GDPR — Article 9 — Special Category Data** `gambling_spend` and `had_adult_entertainment_raw`
may reveal information about a person's habits, health, or private life. Processing such data
for automated credit decisions requires explicit consent or another Article 9(2) legal basis,
which a standard credit application does not provide.

**GDPR — Article 22 — Automated Decision-Making** Using sensitive spending categories as direct
model features in an automated credit scoring system requires either explicit consent or a
suitable legal basis. Applicants must be informed and have the right to request human review.

**GDPR — Article 5(1)(c) — Data Minimisation** Including `gambling_spend` and
`had_adult_entertainment_raw` as raw model features goes beyond what is strictly necessary
for credit risk assessment, violating the minimisation principle.


In this notebook, additional red_flag_spending column is added. This approach:
- **Removes** raw sensitive values from model input features
- **Preserves** the risk signal in a non-sensitive, aggregated form
- **Satisfies** data minimisation — the model only sees 0 or 1, not the raw spend
- **Reduces** Article 9 exposure — a binary flag is far less likely to reveal
  sensitive characteristics than the raw figures


#### Recommended Actions

- **Data Engineering:** Drop `gambling_spend` and `had_adult_entertainment_raw` from
  model feature sets and replace with `red_flag_spending`.
- **Governance Officer:** Document the threshold logic and legal basis for deriving
  `red_flag_spending` in the data processing record (Article 30 ROPA).
- **Governance Officer:** Confirm whether explicit consent or a legitimate interest
  assessment is in place to justify collecting these fields at all.

## Final form

In [12]:
df_clean.iloc[:10,:20].head()

,_id,full_name,email,ssn,ip_address,gender,date_of_birth,zip_code,annual_income,income_imputed,credit_history_months,credit_history_months_flag,debt_to_income,dti_flag,savings_balance,loan_approved,interest_rate,approved_amount,rejection_reason,processing_timestamp
0,app_200,TKN_905902,c5e26f450af27ab8c4df90a9992d34e962fcd7e8603853...,***-**-4340,192.168.*.*,Male,2001.0,100**,68677.0 - 77323.0,False,22.0 - 24.0,False,0.0 - 0.0,False,30004.0 - 32420.0,False,nan - nan,nan - nan,algorithm_risk_score,2024-01-15
1,app_037,TKN_451201,78a73ea33f2ac43378b227b701f6dc4a075b145dd4f607...,***-**-4784,10.1.*.*,Male,1992.0,100**,72292.0 - 83708.0,False,50.0 - 52.0,False,0.0 - 0.0,False,8856.0 - 26974.0,False,nan - nan,nan - nan,algorithm_risk_score,NaT
2,app_215,TKN_536334,189ad918bd469aec8f24a2359d9cf53c21232d784e4cb3...,***-**-5178,10.240.*.*,Male,1989.0,100**,53942.0 - 68058.0,False,39.0 - 43.0,False,0.0 - 0.0,False,34160.0 - 41658.0,True,4.0 - 4.0,56504.0 - 61496.0,NaN,NaT
3,app_024,TKN_316527,5743dfcbf5f5c32908f05f6d4b0f190115a2c735036d81...,***-**-1833,192.168.*.*,Male,1983.0,100**,91516.0 - 114484.0,False,69.0 - 71.0,False,0.0 - 0.0,False,-3436.0 - 3436.0,True,4.0 - 4.0,33539.0 - 34461.0,NaN,NaT
4,app_184,TKN_036972,6aadf96302e9d40b2f34c32ddea898234fe184b02166bf...,***-**-2475,172.29.*.*,Male,1999.0,100**,42667.0 - 71333.0,False,13.0 - 15.0,False,0.0 - 0.0,False,30707.0 - 32819.0,False,nan - nan,nan - nan,algorithm_risk_score,2024-01-15


# Section 5 - Accuracy 
## #13 Age Plausibility, #14 Timestamp Plausibility, #15 Cross-Field Plausibility

### Compliance Analysis — Accuracy

#### Applicable Regulations

**GDPR — Article 5(1)(d) — Accuracy**
All three accuracy issues involve data that exists and was successfully parsed, but whose values are suspicious or internally contradictory. Unlike completeness issues where data is simply absent, accuracy issues are more dangerous — the data appears valid on the surface but contains hidden errors that could silently corrupt any model or decision built on top of it.

**EU AI Act — Article 10(3) — Data and Data Governance**
Credit scoring is a high-risk AI application under Annex III of the EU AI Act. Implausible ages, impossible timestamps, and internally contradictory field combinations are all forms of data error that directly violate the quality requirements for high-risk AI training data. A model trained on these records would learn from corrupted inputs, potentially producing systematically incorrect or discriminatory credit decisions.

**EU AI Act — Article 14(1) — Human Oversight**
None of the three issues identified in this section can be safely corrected automatically — each requires human investigation to determine the true value. Flagging without altering is therefore not just a conservative choice but a legal requirement under the AI Act's human oversight obligations.

---

#### Issue Breakdown

**#13 — Age Plausibility** — Applicants younger than 18 cannot legally enter into a credit agreement, and ages above 85 are statistically implausible for this dataset and likely indicate a data entry error such as a wrong century (e.g. `1923` instead of `2023`). Although no violations were found in the current dataset, the check is formally documented here as a standing governance control — any future record with an age outside the valid range `[18, 85]` must be flagged and reviewed before use in any credit decision.

**#14 — Timestamp Plausibility** — Any processing timestamp predating NovaCred's founding year (2020) or postdating the audit date (March 1, 2026) is factually impossible and indicates either a system clock fault or a data entry error. Such timestamps cannot be trusted as evidence of when a decision was made, undermining the audit trail for those records.

**#15 — Cross-Field Plausibility** — Cross-field violations are the most subtle and dangerous accuracy issue in this dataset — they cannot be detected by inspecting any single field in isolation. Three specific contradictions were checked: decision fields inconsistent with the approval outcome, negative savings balances, and rejection reasons contradicted by the applicant's actual DTI ratio. Any of these contradictions, if undetected, would silently corrupt the model's understanding of the relationship between financial features and credit outcomes.

---

#### Consequences if Unresolved
- Applicants with implausible ages could be incorrectly included or excluded from credit eligibility assessments, constituting an unfair automated decision under **GDPR Article 22**.
- Records with impossible timestamps cannot be used as evidence in any regulatory audit or legal challenge, exposing NovaCred to liability under the EU AI Act's record-keeping requirements.
- Cross-field contradictions fed into the credit scoring model would corrupt the learned relationships between features, producing unreliable predictions for the entire applicant population — not just the affected records.

---

#### Recommended Actions
- **Data Engineering Team**: Implement age range validation at ingestion — reject any application where the derived age falls outside `[18, 85]`.
- **Data Engineering Team**: Enforce timestamp validation at ingestion and at every decision write-back — timestamps must fall within the valid operating window and must be recorded automatically at the point of every credit decision.
- **Data Engineering Team**: Implement cross-field validation rules at ingestion — approved applications must always have `interest_rate` and `approved_amount`, rejected applications must always have `rejection_reason`, and `savings_balance` must always be non-negative.
- **Governance Officer (this report)**: The 1 record flagged with `neg_savings_flag = True` (`app_290`, `savings_balance = -5000`) is hereby flagged for mandatory manual review and must be excluded from all model training until the correct value is confirmed. All other cross-field flags with zero violations are formally documented as standing governance controls for future data cycles.

# Section 6 - Timeliness

## #15 & #16 - Imposible or old timestamps

### Applicable Regulations

**GDPR — Article 5(1)(d) — Accuracy** Records with implausible timestamps (app_179: 2026-03-15,
app_051: 2027-01-20) contain metadata that does not reflect reality, violating the requirement
to keep personal data accurate and up to date.

**GDPR — Article 5(1)(e) — Storage Limitation** 59 records predate the 2-year retention window
(threshold: 2022-03-01). Retaining them without a documented lawful basis breaches the principle
that data must not be kept longer than necessary.

**GDPR — Article 25 — Data Protection by Design** The presence of future-dated and stale records
indicates that timestamp validation was never built into the ingestion pipeline, violating the
requirement to implement data quality controls by design.

**EU AI Act — Article 10 — Data Governance** Implausible and stale records must not be used to
train high-risk AI models. Credit scoring models must be trained on data free from errors
to the best extent possible.

---

### Consequences if Unresolved

- Implausible timestamps introduce temporally incoherent signals into model training,
  producing credit decisions that cannot be legally defended.
- Stale records may reflect outdated financial circumstances, leading to inaccurate
  credit scores for affected applicants.
- 438 records with missing timestamps mean NovaCred cannot confirm whether the majority
  of its dataset falls within a lawful retention window.

---

### Recommended Actions

- **Data Engineering:** Add timestamp validation at ingestion — reject records with
  future-dated or missing timestamps at the pipeline level.
- **Data Engineering:** Backfill or investigate the 438 records with missing timestamps
  to determine whether they fall within the lawful retention window.
- **Governance Officer:** Exclude all `timestamp_implausible = True` and
  `timestamp_stale = True` records from model training immediately.
- **Governance Officer:** Document a lawful retention basis for any stale records
  that must be kept, or schedule them for deletion.

## Regulatory Articles Reference — Section 6 (Bias Analysis)

The following articles govern the bias findings in this section. Articles already introduced in Sections 1–5 (GDPR Art. 5, Art. 33, EU AI Act Art. 12, Art. 14) are not repeated here — refer to the consolidated reference at the top of the notebook.

| Article | Regulation | Title | Relevance to Bias Analysis |
|---------|------------|-------|----------------------------|
| Article 22 | GDPR | Automated Decision-Making | Automated credit decisions producing systematically worse outcomes for a protected group (gender, age) violate the right not to be subject to discriminatory automated processing. |
| Article 9 | EU AI Act | Risk Management | High-risk AI systems must implement a risk management process that explicitly identifies and mitigates foreseeable discriminatory outcomes, including those arising from feature interactions. |
| Article 10 | EU AI Act | Data and Data Governance | Training data must be examined for bias with respect to protected characteristics; bias monitoring must continue throughout the system's operational lifetime. |
| Dir. 2000/78/EC | EU Directive | Employment Equality Directive | Prohibits age-based discrimination in access to goods and services; applied by case law to financial products including credit. |
| Dir. 2004/113/EC | EU Directive | Equal Treatment Directive | Prohibits sex-based discrimination in access to goods and services, including financial products; covers both direct and indirect (proxy) discrimination. |

> **Note on fines** — GDPR violations under Article 22 carry a maximum penalty of **€20 million or 4% of global annual turnover**, whichever is higher. This ceiling applies to each of the bias findings documented below and is not repeated in each sub-section.

# Section 6 - Biases

## 1. Gender bias in approvals

Analysis of 497 applications (250 female, 247 male) reveals a systematic gender gap in NovaCred's credit scoring algorithm. Female applicants were approved at **52.0%** compared to **64.4%** for male applicants, which is a gap of **12.4 percentage points**. The resulting Disparate Impact (DI) ratio is **0.808**, sitting just above the legal four-fifths threshold of 0.80, but well within a range that warrants regulatory concern given the scale of the absolute gap. Critically, this cannot be attributed to financial risk differences: female applicants carry comparable income and credit history to male applicants, meaning the model is penalising gender itself.

### Regulatory Violations
- **GDPR Article 22** — restricts automated decision-making that discriminates on protected grounds.
- **EU AI Act Article 10** — mandates bias monitoring and mitigation in high-risk AI systems; credit scoring is explicitly listed as high-risk.

> Proximity to the four-fifths threshold is not a safe harbour. Regulators treat borderline DI ratios alongside the absolute approval gap as evidence of systemic bias.

### Consequences
Fines of up to **€20 million or 4% of global annual turnover** under GDPR, supervisory intervention, and civil liability from affected applicants.

### Recommended Actions
1. Full **model audit** to identify which features drive the gender gap.
2. **SHAP value analysis** to quantify each variable's contribution to approval decisions.
3. **Bias mitigation** via training data reweighting or threshold calibration by gender.
4. **Ongoing DI monitoring** as part of model governance.



## 2. Gender Bias in Approved Loan Amounts

Even among the 289 approved applicants, the algorithm continues to disadvantage women. Female applicants received a mean approved amount of **$46,469** compared to **$49,226** for male applicants — a gap of **$2,757**. The median tells the same story: **$48,000** for women vs **$51,000** for men. This confirms that gender bias is not limited to who gets approved, but extends to how much they receive.

Section 1 showed that women are approved at a lower rate (52.0% vs 64.4%, DI = 0.808). This section reveals the discrimination compounds further for those who do clear that hurdle — approved women still receive systematically less than approved men. Together, the two findings suggest the algorithm disadvantages women at two distinct decision points: the approval gate and the amount assigned. This two-layer bias significantly strengthens the case for regulatory intervention.

### Regulatory Violations
- **GDPR Article 22** — the loan amount constitutes a significant automated decision affecting individuals' financial lives, and must not be influenced by protected characteristics.
- **EU AI Act Article 10** — bias in output variables beyond the binary approval decision falls within the scope of required bias monitoring for high-risk systems.

> A $2,757 mean gap and a $3,000 median gap among approved applicants is material — at scale, this represents substantial cumulative financial harm to female customers.

### Consequences
In addition to the GDPR fine exposure noted in §1, NovaCred faces consumer protection claims and potential class action liability from female applicants who received systematically lower amounts than equally or less qualified male peers.

### Recommended Actions
1. Extend the model audit to cover **amount assignment**, not just the binary approval decision.
2. Test whether amount-determining features (e.g. income, DTI) are being applied consistently across genders.
3. Consider **post-hoc correction mechanisms** to equalise approved amounts for equivalent risk profiles.
4. Include loan amount distributions in **ongoing fairness reporting** by gender.



## 3. Age-Based Bias

Analysis of approval rates across age groups reveals a clear and systematic pattern of age discrimination in NovaCred's algorithm. The 18-29 group has the lowest approval rate at **41.5%** (n=82), nearly **27 percentage points** below the 40-49 peak of **68.8%** (n=138). The 30-39 group sits at **56.7%** (n=171), 50-59 at **56.5%** (n=62), and 60+ at **65.1%** (n=43). The algorithm consistently favours middle-aged applicants, with no financial justification for penalising younger or older applicants at this scale.

### Regulatory Violations
- **GDPR Article 22** — age is a protected characteristic; automated decisions producing systematically worse outcomes for specific age groups constitute unlawful discrimination.
- **EU AI Act Article 10** — requires high-risk AI systems to be tested for bias across all protected attributes, including age, prior to and during deployment.
- **EU Employment Equality Directive (2000/78/EC)** — prohibits age-based discrimination in access to goods and services, including financial products.

### Consequences
The same GDPR fine ceiling applies. Beyond financial penalties, NovaCred faces regulatory scrutiny from financial supervisory authorities and potential claims from young applicants who were disproportionately rejected without objective justification.

### Recommended Actions
1. Audit which model features correlate with age and assess whether they serve as **proxies for age discrimination**.
2. Compute **DI ratios across all age groups** using the youngest or most disadvantaged group as the reference.
3. Apply **age-stratified fairness constraints** during model training or post-processing.
4. Ensure sample sizes for smaller age groups (e.g. 60+, n=43) are expanded in future data collection to support robust fairness testing.

### Conclusion
The 27 percentage point approval gap between the youngest and most favoured age group is too large to be explained by legitimate risk differentials alone. Without corrective action, NovaCred is systematically denying credit access to young adults and exposing itself to significant legal and reputational risk.



## 4.Interaction Effect: Gender × Age

When gender and age are analysed together, the bias revealed in previous sections intensifies sharply. Among the 18-29 age group, female applicants are approved at just **30.4%** compared to **55.6%** for male peers, which is a DI ratio of **0.548**, far below the legal threshold of 0.80 and the most severe violation identified in this analysis. The 50-59 group also shows a notable gap (Female **50.0%** vs Male **62.1%**, DI = **0.806**). Only the 60+ group shows a reversal, with women marginally favoured (DI = **1.095**). The overall gender DI of 0.808 reported in Section 1 masks how severe the discrimination is for specific subgroups, particularly young women.

### Regulatory Violations
- **GDPR Article 22** — intersectional discrimination, where the combination of two protected characteristics produces disproportionate harm, falls squarely within the scope of unlawful automated decision-making.
- **EU AI Act Article 10** — explicitly requires bias evaluation across intersectional subgroups, not just individual protected attributes in isolation.
- **EU AI Act Article 9** — risk management obligations require identification and mitigation of foreseeable discriminatory outcomes, including those emerging from feature interactions.

### Consequences
The same GDPR fine ceiling applies. Additionally, the severity of the DI violation for young women (0.548) significantly strengthens the case for enforcement action, increasing NovaCred's exposure to both individual and collective redress claims from this group specifically.

### Recommended Actions
1. Perform a full **intersectional fairness audit** across all combinations of protected attributes.
2. Use **SHAP interaction values** to identify which feature combinations are driving the compounded penalty for young female applicants.
3. Apply **subgroup-specific fairness constraints** during model retraining to prevent intersectional violations from being hidden by aggregate metrics.
4. Report DI ratios at the intersectional level — not just overall — in all future fairness disclosures.

### Conclusion
A DI of 0.548 for young female applicants represents a critical compliance failure. The fact that the overall gender DI of 0.808 appears borderline acceptable makes this finding particularly dangerous — aggregate metrics actively conceal the severity of harm experienced by specific subgroups. Intersectional analysis must be a standard component of NovaCred's fairness monitoring going forward.



## 5. Proxy Discrimination

A proxy variable is a non-protected attribute that is highly correlated with a protected characteristic, introducing indirect discrimination even when that characteristic is not explicitly used by the model. Two potential proxies were investigated: **ZIP code** and **annual income**.

---

### 5.1 ZIP Code as a Proxy for Gender

The dataset contains three ZIP code prefixes with starkly different demographic compositions and approval rates. ZIP 902 (80% female, n=229) has an approval rate of **51.5%**, ZIP 300 (44% female, n=18) sits at **55.6%**, and ZIP 100 (24% female, n=250) reaches **64.4%**. The correlation between the female ratio of an area and its approval rate is **-0.935** — near perfect negative correlation. Areas with more female applicants are systematically approved at lower rates, meaning geographic location is functioning as a disguised gender filter.

### 5.2 Income as a Proxy — Does the Gender Gap Persist After Controlling for Income?

If income differences explained the gender gap, the disparity should disappear within each income quartile. It does not. In Q1 (lowest income) the DI is **0.618** — a clear violation. Q2 yields a DI of **0.711**, also a violation. Only Q3 shows parity (DI = **1.033**), and Q4 remains borderline at **0.840**. Critically, female applicants have a slightly higher mean income (**$82,306**) than male applicants (**$83,089**), confirming that income does not explain the gap — the model is penalising gender beyond any legitimate financial risk differential.

### 5.3 Debt-to-Income Ratio as a Proxy

The most severe finding in the entire analysis emerges here. Women in Q1 — the lowest debt-to-income quartile, representing the **best financial profiles** — are approved at only **45.9%** versus **80.7%** for equivalent men, yielding a DI of **0.569**. Q2 (DI = **0.909**) and Q3 (DI = **0.853**) are marginal, and only Q4 approaches parity (DI = **0.967**). The algorithm is most discriminatory precisely where it should be most favourable — against financially responsible women.

---

### Regulatory Violations, Consequences & Recommended Actions

**On ZIP code (5.1):** Under **GDPR Article 22** and the **Equal Treatment Directive (2004/113/EC)**, using a variable that encodes a protected characteristic constitutes indirect discrimination regardless of intent. NovaCred cannot claim gender was absent from the model when a near-perfect gender proxy was present. ZIP code should be removed as a model input immediately pending a full proxy audit.

**On income (5.2):** Under **EU AI Act Article 10**, model inputs must be assessed for proxy effects prior to deployment. The persistence of the gender gap across income quartiles — despite women earning comparably — demonstrates that income is not neutralising the bias but potentially masking it. Income quartile stratification must be included in all future fairness reporting.

**On debt-to-income ratio (5.3):** This finding represents the most direct violation of **GDPR Article 22** in the entire dataset. A DI of 0.569 among the most creditworthy applicants is evidence that the model is inverting its own stated objective of evaluating financial risk. This exposes NovaCred to the highest level of regulatory and litigation risk of any finding in this report.

**Across all three proxies**, the GDPR fine ceiling noted throughout this section applies in full — proxy discrimination carries the same legal weight as direct discrimination under EU law. Recommended actions include a full **feature correlation audit** against all protected attributes, **fairness-aware feature selection**, subgroup-level DI reporting across all financial variables, and immediate review of any feature whose removal materially improves fairness metrics without degrading predictive performance.

### Conclusion
Proxy discrimination is the most insidious finding in this analysis because it can persist even after protected attributes are formally excluded from the model. The -0.935 ZIP-gender correlation and the 0.569 DI among low-DTI women together demonstrate that NovaCred's algorithm is encoding gender through multiple indirect channels simultaneously. Remediation must go beyond removing explicit protected attributes — it requires a systematic audit of every input variable for discriminatory proxy potential.



# Audit Summary

### All Identified Issues

| # | Issue | Dimension | Severity | Key Regulations |
|---|-------|-----------|----------|-----------------|
| 1 | Duplicate Records | Uniqueness | 🟠 HIGH | GDPR Art. 5(1)(c), 5(1)(d); EU AI Act Art. 10 |
| 2 | Conflicting SSNs | Uniqueness | 🔴 CRITICAL | GDPR Art. 5(1)(d), Art. 33; EU AI Act Art. 10 |
| 3 | Inconsistent Gender Coding | Consistency | 🟡 MEDIUM | GDPR Art. 5(1)(d); EU AI Act Art. 10 |
| 4 | Mixed Date Formats | Consistency | 🟠 HIGH | GDPR Art. 5(1)(d); EU AI Act Art. 10 |
| 5 | Income Stored as Text | Validity | 🟠 HIGH | GDPR Art. 5(1)(d); EU AI Act Art. 10(3), Art. 22 |
| 6 | Negative Credit History | Validity | 🟠 HIGH | GDPR Art. 5(1)(d); EU AI Act Art. 10(3), Art. 14(1) |
| 7 | Impossible DTI Ratio | Validity | 🟠 HIGH | GDPR Art. 5(1)(d); EU AI Act Art. 10(3), Art. 14(1) |
| 8 | Missing Date of Birth | Completeness | 🟡 MEDIUM | GDPR Art. 5(1)(c), 5(1)(d) |
| 9 | Missing Email Address | Completeness | 🟠 HIGH | GDPR Art. 22, Art. 33; EU AI Act Art. 10(3) |
| 10 | Missing SSN | Completeness | 🟠 HIGH | GDPR Art. 22; EU AI Act Art. 10(3) |
| 11 | Missing Timestamps (400 records) | Completeness | 🟠 HIGH | EU AI Act Art. 12, Art. 14(1) |
| 12 | Missing Income (imputed) | Completeness | 🟡 MEDIUM | EU AI Act Art. 10(3) |
| 13 | Age Plausibility | Accuracy | 🟡 MEDIUM | GDPR Art. 5(1)(d), Art. 22; EU AI Act Art. 10(3) |
| 14 | Timestamp Plausibility | Accuracy | 🟡 MEDIUM | EU AI Act Art. 12; GDPR Art. 5(1)(d) |
| 15 | Cross-Field Plausibility | Accuracy | 🟠 HIGH | GDPR Art. 5(1)(d), Art. 22; EU AI Act Art. 10(3) |
| B1 | Gender Bias in Approvals (DI = 0.808) | Bias | 🔴 CRITICAL | GDPR Art. 22; EU AI Act Art. 10 |
| B2 | Gender Bias in Loan Amounts ($2,757 gap) | Bias | 🔴 CRITICAL | GDPR Art. 22; EU AI Act Art. 10 |
| B3 | Age-Based Discrimination (27pp gap) | Bias | 🔴 CRITICAL | GDPR Art. 22; EU AI Act Art. 10; Dir. 2000/78/EC |
| B4 | Intersectional Bias: Gender × Age (DI = 0.548) | Bias | 🔴 CRITICAL | GDPR Art. 22; EU AI Act Art. 9, Art. 10 |
| B5 | ZIP Code as Gender Proxy (r = −0.935) | Bias | 🔴 CRITICAL | GDPR Art. 22; Equal Treatment Dir. 2004/113/EC |
| B6 | Income as Gender Proxy | Bias | 🟠 HIGH | GDPR Art. 22; EU AI Act Art. 10 |
| B7 | DTI Ratio as Gender Proxy (DI = 0.569 in Q1) | Bias | 🔴 CRITICAL | GDPR Art. 22; EU AI Act Art. 10 |

---

### Regulatory Articles Reference

| Article | Regulation | Title | Summary |
|---------|------------|-------|---------|
| Art. 5(1)(b) | GDPR | Purpose Limitation | Data collected for one purpose must not be silently reused for another. |
| Art. 5(1)(c) | GDPR | Data Minimisation | Data must be limited to what is necessary; inadequate records must not be processed. |
| Art. 5(1)(d) | GDPR | Accuracy | Inaccurate or incomplete personal data must be rectified or erased without delay. |
| Art. 4(5) | GDPR | Pseudonymisation | Data must be processed so it cannot be attributed to a subject without separate additional information. |
| Art. 22 | GDPR | Automated Decision-Making | Individuals have the right not to be subject to solely automated decisions with significant effects. |
| Art. 33 | GDPR | Data Breach Notification | Personal data breaches must be reported to the supervisory authority within 72 hours. |
| Art. 9 | EU AI Act | Risk Management | High-risk AI systems must identify and mitigate foreseeable discriminatory outcomes. |
| Art. 10 | EU AI Act | Data and Data Governance | Training data for high-risk AI must be relevant, representative, and free from errors. |
| Art. 12 | EU AI Act | Record-Keeping | High-risk AI systems must automatically log events to enable full auditability of decisions. |
| Art. 14(1) | EU AI Act | Human Oversight | High-risk AI systems must allow natural persons to effectively oversee them during use. |
| Dir. 2000/78/EC | EU Directive | Employment Equality Directive | Prohibits age-based discrimination; extended by case law to financial services access. |
| Dir. 2004/113/EC | EU Directive | Equal Treatment Directive | Prohibits sex-based discrimination in access to goods and services including financial products. |

---

### Conclusion

The data quality issues in Sections 1–5 are fixable. Missing timestamps, inconsistent encodings, 
impossible field values are pipeline problems, and pipeline problems have engineering solutions. 
Seven people were approved with no contact details on file. Four with no 
verified identity. Those decisions happened, but they need to be reviewed.

Section 6 is harder to resolve. A gender DI of 0.808 can be argued away as borderline, until you 
break it down by age and it becomes 0.548 for women under 30, or until you look at the DTI quartiles 
and find that the most financially responsible women in the dataset are also the most likely to be 
rejected. That is the model systematically working against the people it should 
be most willing to approve. The ZIP code correlation of -0.935 means geography is doing what the 
model is not supposed to do explicitly. Removing the gender field would not fix this.

The path forward is straightforward even if it is not quick: freeze the model for retraining, 
implement the ingestion controls described in each section, and set up intersectional DI monitoring 
before the next deployment. The EU AI Act obligations are already live. The sooner this is treated 
as an operational priority rather than a compliance checkbox, the less likely it ends with a 
supervisory inquiry.